# example of reading one trace

In [1]:
using Pkg

cd(@__DIR__)
Pkg.activate("../")
using Dates, SonifSismo, CairoMakie

  Activating project at `c:\Users\lucie\Desktop\sonifSismo.jl-main`


In [2]:
archive = DataArchive("C:/Users/lucie/Desktop/mtFujiContinuous/mtFujiContinuous")

DataArchive("C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\catalog\\h2008", :JST, Hour(9))

In [3]:

available_stations(archive)
available_channels(archive)

33-element Vector{ChannelAvailability}:
 ChannelAvailability("EV", "FJO", "", "E", [Date("2008-05-25"), Date("2008-05-26"), Date("2008-05-27"), Date("2008-05-28"), Date("2008-05-29"), Date("2008-05-30"), Date("2008-05-31")], ["C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\25\\EV.FJO..E.20080525.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\26\\EV.FJO..E.20080526.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\27\\EV.FJO..E.20080527.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\28\\EV.FJO..E.20080528.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\29\\EV.FJO..E.20080529.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\30\\EV.FJO..E.20080530.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\31\\EV.FJO..E.2008053

In [4]:
start_time = DateTime(2008, 5, 28, 0, 0, 0)  # UTC
end_time   = DateTime(2008, 5, 28, 23, 59, 59)     # UTC

traces = read_window(
    archive,
    start_time,
    end_time;
    stations="FUJ",
    channels=["wE", "wN", "wU"],
    merge=true,
 #    merge_gaps=:zero,       # or :linear
    processed=false,
    bandpass=(0.5, 15.0),
)

3-element Vector{Seis.AbstractTrace}:
 Seis.Trace(EV.FUJ..wE : delta=0.01, b=0.0, nsamples=8639901)
 Seis.Trace(EV.FUJ..wN : delta=0.01, b=0.0, nsamples=8639901)
 Seis.Trace(EV.FUJ..wU : delta=0.01, b=0.0, nsamples=8639901)

In [5]:
include("C:/Users/lucie/Desktop/sonifSismo.jl-main/examples/sonify_one_trace.jl")

trace_wU = only(t for t in traces if strip(t.sta.cha) == "wU")
trace_wN = only(t for t in traces if strip(t.sta.cha) == "wN")
trace_wE = only(t for t in traces if strip(t.sta.cha) == "wE")


Seis.Trace{Float64,Vector{Float32},Seis.Geographic{Float64}}:
            b: 0.0
        delta: 0.01
 GeogStation{Float64}:
      sta.net: EV
      sta.sta: FUJ
      sta.loc: 
      sta.cha: wE 
     sta.meta: Seis.SeisDict{Symbol, Any}()
 GeogEvent{Float64}:
     evt.time: 2008-05-28T00:00:00
     evt.meta: Seis.SeisDict{Symbol, Any}()
 Trace:
        picks: 0
         meta: mseed_file => "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\28\\EV.FUJ..wE.20080528.mseed"

In [ ]:
# Construction des 3 signaux, un par un
audioE = sonify_trace(               
    trace_wE;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

audioN = sonify_trace(
    trace_wN;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

audioU = sonify_trace(
    trace_wU;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

# Extraction des signaux
sigE = audioE.signal     # On est obligé de faire ça pour récupérer les signaux, car les objets audio sont des tuples (contiennent d'autres informations comme la fréquence d'échantillonnage, etc.)
sigN = audioN.signal
sigU = audioU.signal

# Vérification longueur des signaux (car tous les signaux ne sont pas forcément de la même longueur, à cause des éventuels trous dans les données d'origine)
n = minimum(length.([sigE, sigN, sigU]))

sigE = sigE[1:n]
sigN = sigN[1:n]
sigU = sigU[1:n]

# Combinaison des 3 signaux en un seul
mix = (sigE .+ sigN .+ sigU) ./ 3

mix ./= maximum(abs.(mix))

# Enregistrement du signal combiné dans un fichier WAV
wavwrite(
    mix,
    "seisme_3c.wav";
    Fs=audioE.sample_rate     # Fs = fréquence d'échantillonnage de sortie (la fonction de sonification a déjà choisi une fréquence adaptée)
)
